[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/information_theory/06_information_theory_in_deep_learning/exercises.ipynb)

# Module 06 — Exercises: Information Theory in Deep Learning

Twenty-six solved problems in four tiers. Every problem carries a statement, a one-line
intuition, a stepwise solution, a boxed answer, a key takeaway, and — wherever the answer is
numeric or algorithmic — a code cell that recomputes it and prints the check.

Theorem, proof and example numbers refer to
[first_principles.ipynb](first_principles.ipynb). Symbols follow
[the notation register](../../docs/notation.md): $D_{\mathrm{KL}}(p \parallel q)$ for relative
entropy, $H_{\times}(p, q)$ for cross-entropy, $\mathcal{L}_{\mathrm{NCE}}$ for the
**unnormalized** InfoNCE loss with chance level $\log K$, and every numerical answer carries its
unit — nats for $\ln$, bits for $\log_2$.

The preamble below is shared by every code cell in this notebook.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (7.0, 4.0),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})
rng = np.random.default_rng(0)
np.set_printoptions(precision=4, suppress=True)

EPS = np.finfo(float).eps
NAT_TO_BIT = 1.0 / np.log(2.0)


def entropy(p):
    """Shannon entropy of a discrete distribution, in nats."""
    p = np.asarray(p, dtype=float)
    p = p[p > 0.0]
    return float(-(p * np.log(p)).sum())


def kl(p, q):
    """Kullback-Leibler divergence D(p || q), in nats."""
    p, q = np.asarray(p, dtype=float), np.asarray(q, dtype=float)
    m = p > 0.0
    if np.any(q[m] <= 0.0):
        return np.inf
    return float((p[m] * np.log(p[m] / q[m])).sum())


def mutual_information(joint):
    """Mutual information of a discrete joint given as a 2-D array, in nats."""
    joint = np.asarray(joint, dtype=float)
    px, py = joint.sum(axis=1), joint.sum(axis=0)
    m = joint > 0.0
    return float((joint[m] * np.log(joint[m] / np.outer(px, py)[m])).sum())


print(f"machine epsilon = {EPS:.4e}")
print(f"1 nat = {NAT_TO_BIT:.6f} bits")

machine epsilon = 2.2204e-16
1 nat = 1.442695 bits


## L0 — Concept Checks

### Problem L0.1 — The ELBO gap

**Statement.** A variational autoencoder reports
$\mathcal{L}_{\mathrm{ELBO}} = -104.2$ nats on an example whose true log-likelihood under the
model is $\log p_\theta(x) = -101.7$ nats. What is the inference gap, and what does it measure?

**Intuition.** Theorem 4.1 says the difference between the two numbers is a KL divergence, so the
gap is not a mystery but a named quantity.

**Solution.**

*Step 1.* By Theorem 4.1,
$\log p_\theta(x) = \mathcal{L}_{\mathrm{ELBO}} + D_{\mathrm{KL}}\left(q_\phi(z \mid x) \parallel p_\theta(z \mid x)\right)$.

*Step 2.* Rearranging, the gap is $-101.7 - (-104.2) = 2.5$ nats $= 3.61$ bits.

*Step 3.* It is **inference** error, not model error: the amount by which the amortized encoder
misses the model's own posterior. A better decoder cannot reduce it; a richer variational family
can.

$$
\boxed{D_{\mathrm{KL}}\left(q_\phi \parallel p_\theta(\cdot \mid x)\right) = 2.5 \text{ nats} = 3.61 \text{ bits}}
$$

**Key takeaway.** The ELBO's slack is always a KL divergence, and it names the component to fix.

In [2]:
elbo, log_px = -104.2, -101.7
gap = log_px - elbo
print(f"inference gap = {gap:.4f} nats = {gap * NAT_TO_BIT:.4f} bits")
assert abs(gap - 2.5) < 1e-12

inference gap = 2.5000 nats = 3.6067 bits


### Problem L0.2 — Reading the Information Bottleneck multiplier

**Statement.** In $\mathcal{L}_{\mathrm{IB}} = I(X; Z) - \beta\, I(Z; Y)$, minimized over
encoders, describe the optimum as $\beta \to 0$ and as $\beta \to \infty$.

**Intuition.** $\beta$ prices relevance against rate; the two limits switch one term off.

**Solution.**

*Step 1 ($\beta \to 0$).* The objective reduces to minimizing $I(X; Z)$, whose minimum $0$ is
attained by any $Z$ independent of $X$ — a constant. Maximally compressed and useless.

*Step 2 ($\beta \to \infty$).* Relevance dominates, and by the data-processing inequality
$I(Z; Y) \le I(X; Y)$, so the optimum attains $I(Z; Y) = I(X; Y)$ and $Z$ is a **minimal
sufficient statistic** of $X$ for $Y$ — minimal because at any large finite $\beta$ the rate term
still breaks ties.

*Step 3 (in between).* By Theorem 4.7 the optimum sits where the information curve has slope
$1/\beta$, and by Proof 5.6 Step 5 nothing below $\beta = 1$ is non-trivial.

$$
\boxed{\beta \to 0: \; Z \perp X, \; I(X;Z) = 0; \qquad \beta \to \infty: \; Z \text{ minimal sufficient for } Y}
$$

**Key takeaway.** $\beta$ is an exchange rate in nats of relevance per nat of rate; the two limits
bracket every useful representation.

### Problem L0.3 — Chance-level InfoNCE

**Statement.** A contrastive model with $K = 128$ candidates reports
$\mathcal{L}_{\mathrm{NCE}} = 4.85$ nats. Is it learning anything?

**Intuition.** Under the unnormalized convention of Definition 3.5, chance is $\log K$, not zero.

**Solution.**

*Step 1.* Chance level is $\log 128 = 4.852030$ nats.

*Step 2.* The reported $4.85$ is indistinguishable from chance.

*Step 3.* By Theorem 4.9 the certified information is
$\log K - \mathcal{L}_{\mathrm{NCE}} = 4.852030 - 4.85 = 0.002030$ nats — essentially nothing.

$$
\boxed{\mathcal{L}_{\mathrm{NCE}} \approx \log K \implies \text{certified } I \approx 0.002 \text{ nats}}
$$

**Key takeaway.** Compare a contrastive loss to $\log K$, never to zero — the same discipline as
comparing a classification loss to the log of the number of classes.

In [3]:
K, loss_nce = 128, 4.85
chance = np.log(K)
print(f"chance level log K   = {chance:.6f} nats")
print(f"reported loss        = {loss_nce:.6f} nats")
print(f"certified bound      = {chance - loss_nce:.6f} nats = {(chance - loss_nce) * NAT_TO_BIT:.6f} bits")
assert abs(chance - 4.852030263919617) < 1e-12
assert 0.0 < chance - loss_nce < 0.01

chance level log K   = 4.852030 nats
reported loss        = 4.850000 nats
certified bound      = 0.002030 nats = 0.002929 bits


### Problem L0.4 — Two-part codes and model selection

**Statement.** Model A needs $200$ bits to describe and codes the dataset in $8{,}400$ bits.
Model B needs $1{,}500$ bits and codes it in $7{,}000$. Which does MDL prefer, and what is the
equivalent likelihood-ratio statement?

**Intuition.** Definition 3.6 adds the two parts; the winner is whichever total is shorter.

**Solution.**

*Step 1.* $L_A = 200 + 8{,}400 = 8{,}600$ bits and $L_B = 1{,}500 + 7{,}000 = 8{,}500$ bits.

*Step 2.* MDL prefers **B** by $100$ bits: $1{,}300$ extra bits of complexity buy $1{,}400$ bits
of data compression.

*Step 3.* Since $L(\mathcal{D} \mid M) = -\log_2 p(\mathcal{D} \mid M)$, the comparison is a
Bayes factor with prior $\pi(M) \propto 2^{-L(M)}$:
$\log_2 \frac{p(\mathcal{D} \mid B)\pi(B)}{p(\mathcal{D} \mid A)\pi(A)} = 1400 - 1300 = 100$ bits.

$$
\boxed{L_B = 8{,}500 \lt L_A = 8{,}600 \text{ bits} \implies \text{choose B, by } 100 \text{ bits}}
$$

**Key takeaway.** MDL is Bayesian model selection with the prior written as a code; "complexity
penalty" and "prior" are the same object in different units.

In [4]:
L_A = 200 + 8400
L_B = 1500 + 7000
print(f"L_A = {L_A} bits,  L_B = {L_B} bits,  margin = {L_A - L_B} bits")
print(f"Bayes-factor reading: {(8400 - 7000)} - {(1500 - 200)} = {(8400 - 7000) - (1500 - 200)} bits")
assert L_B < L_A and L_A - L_B == 100

L_A = 8600 bits,  L_B = 8500 bits,  margin = 100 bits
Bayes-factor reading: 1400 - 1300 = 100 bits


### Problem L0.5 — Half a bit per halving

**Statement.** For a Gaussian source under squared error, how much extra rate does halving the
distortion cost?

**Intuition.** Theorem 4.4 makes $R$ logarithmic in $1/D$, so a constant factor in $D$ is a
constant increment in $R$.

**Solution.**

*Step 1.* $R(D) = \tfrac12 \log(\sigma^2/D)$, so
$R(D/2) - R(D) = \tfrac12 \log 2$.

*Step 2.* In nats that is $0.346574$; in bits it is exactly $\tfrac12$.

$$
\boxed{R(D/2) - R(D) = \tfrac12 \log 2 = 0.346574 \text{ nats} = 0.5 \text{ bits}}
$$

**Key takeaway.** Half a bit per halving of the mean squared error, independent of $\sigma^2$ and
of where on the curve you start — the information-theoretic form of "6 dB per bit".

In [5]:
sigma2 = 3.7
for D in (0.8, 0.4, 0.2, 0.1):
    print(f"D = {D:.2f}: R = {0.5 * np.log(sigma2 / D):.6f} nats = {0.5 * np.log2(sigma2 / D):.6f} bits")
steps = [0.5 * np.log(sigma2 / D) for D in (0.8, 0.4, 0.2, 0.1)]
increments = np.diff(steps)
print(f"increments = {increments}  (predicted 0.5*log 2 = {0.5 * np.log(2):.6f})")
assert np.allclose(increments, 0.5 * np.log(2))

D = 0.80: R = 0.765738 nats = 1.104727 bits
D = 0.40: R = 1.112312 nats = 1.604727 bits
D = 0.20: R = 1.458885 nats = 2.104727 bits
D = 0.10: R = 1.805459 nats = 2.604727 bits
increments = [0.3466 0.3466 0.3466]  (predicted 0.5*log 2 = 0.346574)


### Problem L0.6 — Nats per token to bits per byte

**Statement.** A language model reaches a validation loss of $2.20$ nats per token on a
tokenizer averaging $4.0$ bytes per token. What is its compression rate in bits per byte?

**Intuition.** The loss is already a code length; only the units change.

**Solution.**

*Step 1.* $2.20 / \log 2 = 3.173929$ bits per token.

*Step 2.* Divide by bytes per token: $3.173929 / 4.0 = 0.793482$ bits per byte.

$$
\boxed{0.793482 \text{ bits per byte}}
$$

**Key takeaway.** Bits per byte is the only cross-tokenizer-comparable number; a nats-per-token
loss is meaningless without the tokenizer beside it.

In [6]:
loss_nats, bytes_per_token = 2.20, 4.0
bits_per_token = loss_nats * NAT_TO_BIT
bpb = bits_per_token / bytes_per_token
print(f"{loss_nats} nats/token = {bits_per_token:.6f} bits/token = {bpb:.6f} bits/byte")
print(f"compression ratio against raw 8-bit bytes: {8 / bpb:.4f}x")
assert abs(bpb - 0.79348227248893) < 1e-12

2.2 nats/token = 3.173929 bits/token = 0.793482 bits/byte
compression ratio against raw 8-bit bytes: 10.0821x


## L1 — Foundations

### Problem L1.1 — Derive the ELBO two ways

**Statement.** Derive
$\log p_\theta(x) \ge \mathbb{E}_{q_\phi}\left[\log p_\theta(x \mid z)\right] - D_{\mathrm{KL}}\left(q_\phi(z \mid x) \parallel p(z)\right)$
(a) by Jensen's inequality and (b) by an exact identity, and state when the bound is tight.

**Intuition.** Jensen tells you a bound exists; the identity tells you what it costs.

**Solution.**

*Step 1 (Jensen).* Insert $q_\phi$ inside the marginalization:

$$
\log p_\theta(x) = \log \mathbb{E}_{q_\phi(z \mid x)}\left[\frac{p_\theta(x, z)}{q_\phi(z \mid x)}\right] \ge \mathbb{E}_{q_\phi}\left[\log \frac{p_\theta(x, z)}{q_\phi(z \mid x)}\right].
$$

Factoring $p_\theta(x, z) = p_\theta(x \mid z)p(z)$ splits the right side into the reconstruction
term and $-D_{\mathrm{KL}}\left(q_\phi(z \mid x) \parallel p(z)\right)$.

*Step 2 (exact identity).* Factor the other way,
$p_\theta(x, z) = p_\theta(z \mid x)p_\theta(x)$:

$$
\mathbb{E}_{q_\phi}\left[\log \frac{p_\theta(x, z)}{q_\phi(z \mid x)}\right] = \log p_\theta(x) - D_{\mathrm{KL}}\left(q_\phi \parallel p_\theta(\cdot \mid x)\right).
$$

No inequality was used; the bound now follows from $D_{\mathrm{KL}} \ge 0$.

*Step 3 (tightness).* Equality holds if and only if
$q_\phi(z \mid x) = p_\theta(z \mid x)$ almost everywhere. Route (b) is the more useful derivation
because it *names* the slack — this is Proof 5.1.

$$
\boxed{\log p_\theta(x) - \mathcal{L}_{\mathrm{ELBO}} = D_{\mathrm{KL}}\left(q_\phi(z \mid x) \parallel p_\theta(z \mid x)\right) \ge 0}
$$

**Key takeaway.** Every variational method in this module follows the second pattern: produce a
bound whose gap is a KL, so that tightening it is itself a learning problem with a known optimum.

In [7]:
p_z = np.array([0.5, 0.5])
p_x_given_z = np.array([[0.75, 0.25], [0.25, 0.75]])
p_a = float(p_z @ p_x_given_z[:, 0])
posterior = p_z * p_x_given_z[:, 0] / p_a
print("checking the identity on random encoders (Example 6.1 model)")
worst = 0.0
for _ in range(2000):
    q = rng.dirichlet(np.ones(2))
    elbo_q = float((q * np.log(p_x_given_z[:, 0])).sum()) - kl(q, p_z)
    worst = max(worst, abs(elbo_q + kl(q, posterior) - np.log(p_a)))
    assert elbo_q <= np.log(p_a) + 1e-12
print(f"max |ELBO + gap - log p(a)| over 2000 random q = {worst:.3e} = {worst / EPS:.1f} * eps")
print(f"the bound never exceeded log p(a) = {np.log(p_a):.6f}")
assert worst < 32 * EPS

checking the identity on random encoders (Example 6.1 model)
max |ELBO + gap - log p(a)| over 2000 random q = 5.551e-16 = 2.5 * eps
the bound never exceeded log p(a) = -0.693147


### Problem L1.2 — Gaussian KL and the arithmetic of posterior collapse

**Statement.** For a diagonal Gaussian posterior
$\mathcal{N}\left(\mu, \operatorname{diag}(\sigma^2)\right)$ against the prior
$\mathcal{N}(0, I)$, write the per-dimension KL, find where it vanishes, and evaluate it at
$(\mu_j, \sigma_j) = (0, 0.99)$ and $(2, 1)$.

**Intuition.** The rate term is a smooth non-negative function with a single flat minimum, and
that flatness is what makes collapse sticky.

**Solution.**

*Step 1 (formula).*

$$
D_j = \tfrac12\left(\mu_j^2 + \sigma_j^2 - 1 - \ln \sigma_j^2\right).
$$

*Step 2 (zero set).* $\partial D_j / \partial \mu_j = \mu_j$ and
$\partial D_j / \partial \sigma_j^2 = \tfrac12\left(1 - 1/\sigma_j^2\right)$ vanish only at
$(\mu_j, \sigma_j^2) = (0, 1)$, where $D_j = 0$. Being a KL it is non-negative, so this is the
global minimum: a dimension carrying no information at all.

*Step 3 (numbers).* At $(0, 0.99)$,
$D_j = \tfrac12\left(0.9801 - 1 - \ln 0.9801\right) = 0.000100$ nats — collapsed for every
practical purpose. At $(2, 1)$, $D_j = \tfrac12(4 + 1 - 1 - 0) = 2$ nats — a well-used dimension.

*Step 4 (second order).* Writing $\sigma_j^2 = 1 + \delta$,
$D_j \approx \tfrac12 \mu_j^2 + \tfrac14 \delta^2$: quadratic in both deviations, so the gradient
vanishes at collapse and the optimizer feels no restoring force.

$$
\boxed{D_j = \tfrac12\left(\mu_j^2 + \sigma_j^2 - 1 - \ln\sigma_j^2\right), \quad D_j = 0 \iff (\mu_j, \sigma_j) = (0, 1)}
$$

**Key takeaway.** Posterior collapse is a flat, attracting optimum of the rate term; free bits
and KL annealing exist to keep the optimizer away from it early in training.

In [8]:
def gaussian_kl(mu, sigma):
    return 0.5 * (mu ** 2 + sigma ** 2 - 1.0 - np.log(sigma ** 2))


for mu, sigma in ((0.0, 0.99), (2.0, 1.0), (0.0, 1.0)):
    print(f"mu = {mu:.2f}, sigma = {sigma:.2f}:  D = {gaussian_kl(mu, sigma):.8f} nats")
assert abs(gaussian_kl(0.0, 0.99) - 0.0001003358535) < 1e-12
assert abs(gaussian_kl(2.0, 1.0) - 2.0) < 1e-12
assert abs(gaussian_kl(0.0, 1.0)) < 1e-15

delta = np.array([0.2, 0.1, 0.05, 0.02])
exact = gaussian_kl(0.0, np.sqrt(1.0 + delta))
approx = 0.25 * delta ** 2
print("\nquadratic approximation near the collapse point")
for d, e, a in zip(delta, exact, approx):
    print(f"  delta = {d:5.2f}:  exact = {e:.8f}   quadratic = {a:.8f}   ratio = {e / a:.4f}")
assert np.all(np.abs(exact / approx - 1.0) < 0.15)

mu = 0.00, sigma = 0.99:  D = 0.00010034 nats
mu = 2.00, sigma = 1.00:  D = 2.00000000 nats
mu = 0.00, sigma = 1.00:  D = 0.00000000 nats

quadratic approximation near the collapse point
  delta =  0.20:  exact = 0.00883922   quadratic = 0.01000000   ratio = 0.8839
  delta =  0.10:  exact = 0.00234491   quadratic = 0.00250000   ratio = 0.9380
  delta =  0.05:  exact = 0.00060492   quadratic = 0.00062500   ratio = 0.9679
  delta =  0.02:  exact = 0.00009869   quadratic = 0.00010000   ratio = 0.9869


### Problem L1.3 — The rate decomposition

**Statement.** Prove
$\mathbb{E}_{p(x)}\left[D_{\mathrm{KL}}\left(q(z \mid x) \parallel p(z)\right)\right] = I(X; Z) + D_{\mathrm{KL}}\left(q(z) \parallel p(z)\right)$
and interpret each term.

**Intuition.** Route the log-ratio through the aggregate posterior and the two halves name
themselves.

**Solution.**

*Step 1.* With $q(z) = \mathbb{E}_{p(x)}\left[q(z \mid x)\right]$,

$$
\log \frac{q(z \mid x)}{p(z)} = \log\frac{q(z \mid x)}{q(z)} + \log\frac{q(z)}{p(z)} .
$$

*Step 2.* Averaging the first term under $p(x)q(z \mid x)$ gives $I(X; Z)$ by definition.

*Step 3.* The second term depends on $z$ alone, whose marginal is $q(z)$, so it averages to
$D_{\mathrm{KL}}\left(q(z) \parallel p(z)\right)$.

*Step 4 (interpretation).* $I(X; Z)$ is the information the code carries; the second KL is pure
waste from the prior hole, and learned or flow-based priors exist to shrink it. This is
Theorem 4.2.

$$
\boxed{R = I(X; Z) + D_{\mathrm{KL}}\left(q(z) \parallel p(z)\right) \ge I(X; Z)}
$$

**Key takeaway.** The VAE's KL term upper-bounds the latent's mutual information, and the excess
is a fixable modelling inefficiency rather than an information cost.

In [9]:
eps_flip = 0.25
encoder = np.array([[1 - eps_flip, eps_flip], [eps_flip, 1 - eps_flip]])
p_data = np.array([0.5, 0.5])
aggregate = p_data @ encoder
for prior, name in ((aggregate, "matched"), (np.array([0.75, 0.25]), "prior hole")):
    R = float(sum(p_data[i] * kl(encoder[i], prior) for i in range(2)))
    I = mutual_information(p_data[:, None] * encoder)
    print(f"{name:11s}: R = {R:.8f}   I(X;Z) = {I:.8f}   KL(q||p) = {kl(aggregate, prior):.8f}"
          f"   residual = {abs(R - I - kl(aggregate, prior)):.2e}")
    assert abs(R - I - kl(aggregate, prior)) < 8 * EPS

matched    : R = 0.13081204   I(X;Z) = 0.13081204   KL(q||p) = 0.00000000   residual = 0.00e+00
prior hole : R = 0.27465307   I(X;Z) = 0.13081204   KL(q||p) = 0.14384104   residual = 0.00e+00


### Problem L1.4 — $\beta$-VAE as constrained optimization

**Statement.** Show that maximizing the reconstruction term subject to a rate constraint
$R \le R_0$ is, by Lagrangian duality, the $\beta$-VAE objective, and interpret $\beta$.

**Intuition.** A multiplier on a budget is always a shadow price; here the budget is bits.

**Solution.**

*Step 1 (constrained problem).* $\max\, (-D)$ subject to $R \le R_0$, with $D$ and $R$ as in
Definition 3.2.

*Step 2 (Lagrangian).* $\mathcal{G} = -D - \beta(R - R_0)$ with $\beta \ge 0$. Since $\beta R_0$
does not depend on the parameters, maximizing $\mathcal{G}$ is maximizing $-D - \beta R$, and
$\beta = 1$ recovers the plain ELBO.

*Step 3 (what $\beta$ is).* At the optimum complementary slackness forces $R = R_0$ whenever
$\beta \gt 0$, and the envelope theorem gives
$\beta = -\partial D^{\star}/\partial R_0$: the marginal distortion saved per extra nat of rate,
that is minus the slope of the rate-distortion curve of Theorem 4.4.

*Step 4 (practice).* Because $R(D)$ is convex and decreasing (Theorem 4.4(a)), large $\beta$
selects low-rate points and small $\beta$ high-rate ones; targeting $R_0$ directly with an
adaptive $\beta$ is usually easier to tune than fixing $\beta$.

$$
\boxed{\max\left\lbrace -D - \beta R \right\rbrace \equiv \max\left\lbrace -D \; : \; R \le R_0 \right\rbrace, \qquad \beta = -\frac{\partial D^{\star}}{\partial R_0}}
$$

**Key takeaway.** $\beta$ is the Lagrange multiplier of a bit budget, and its optimal value is the
local slope of the rate-distortion frontier.

In [10]:
sigma2 = 1.0
R0 = np.linspace(0.1, 1.5, 15)
D_star = sigma2 * np.exp(-2.0 * R0)
beta_envelope = -np.gradient(D_star, R0)
beta_predicted = 2.0 * sigma2 * np.exp(-2.0 * R0)
interior = slice(1, -1)
print("  R0      D*(R0)    -dD*/dR0 (numeric)   2 sigma^2 exp(-2R) (theory)")
for k in (1, 4, 7, 10, 13):
    print(f"{R0[k]:5.2f}  {D_star[k]:9.6f}  {beta_envelope[k]:18.6f}  {beta_predicted[k]:24.6f}")
rel_err = np.abs(beta_envelope[interior] / beta_predicted[interior] - 1.0).max()
print(f"max relative error at interior grid points = {rel_err:.4%}"
      f"   (central differences; the two endpoints use a one-sided rule)")
assert rel_err < 0.01

  R0      D*(R0)    -dD*/dR0 (numeric)   2 sigma^2 exp(-2R) (theory)
 0.20   0.670320            1.349596                  1.340640
 0.50   0.367879            0.740674                  0.735759
 0.80   0.201897            0.406490                  0.403793
 1.10   0.110803            0.223087                  0.221606
 1.40   0.060810            0.122433                  0.121620
max relative error at interior grid points = 0.6680%   (central differences; the two endpoints use a one-sided rule)


### Problem L1.5 — Bits-back: why the KL term is real bits

**Statement.** Show that a bits-back scheme transmits $x$ at expected cost
$\mathbb{E}_q\left[-\log p(x \mid z)\right] + D_{\mathrm{KL}}\left(q(z \mid x) \parallel p(z)\right)$
nats, and explain where the refund comes from.

**Intuition.** Sampling the latent from a queue of message bits rather than from a coin turns the
sampling randomness into payload.

**Solution.**

*Step 1 (naive cost).* Encode $z$ under the shared prior and then $x$ under $p(x \mid z)$:

$$
C_{\text{naive}} = \mathbb{E}_q\left[-\log p(z) - \log p(x \mid z)\right].
$$

*Step 2 (the refund).* The sender decodes $z$ from bits already queued, using $q(z \mid x)$ as
the code. After the receiver reconstructs $x$ it can re-run $q(z \mid x)$, re-encode $z$, and
recover $\mathbb{E}_q\left[-\log q(z \mid x)\right]$ nats of genuine message.

*Step 3 (net).*

$$
C = C_{\text{naive}} - \text{refund} = \mathbb{E}_q\left[-\log p(x \mid z)\right] + D_{\mathrm{KL}}\left(q(z \mid x) \parallel p(z)\right) = -\mathcal{L}_{\mathrm{ELBO}}.
$$

*Step 4 (optimality).* By Theorem 4.1 this exceeds the Shannon optimum $-\log p(x)$ by exactly
the inference gap, and asymmetric numeral systems implement Steps 1 to 3 in practice.

$$
\boxed{\text{bits-back cost} = D + R = -\mathcal{L}_{\mathrm{ELBO}} \ge -\log p(x)}
$$

**Key takeaway.** The KL term is not a metaphor for complexity: a working codec charges exactly
those nats, and the ELBO is the code length it achieves.

In [11]:
q_bad = np.array([0.5, 0.5])
naive = float((q_bad * (-np.log(p_z) - np.log(p_x_given_z[:, 0]))).sum())
refund = float((q_bad * -np.log(q_bad)).sum())
net = naive - refund
shannon = -np.log(p_a)
print(f"naive two-part cost  = {naive:.6f} nats")
print(f"bits-back refund     = {refund:.6f} nats")
print(f"net code length      = {net:.6f} nats")
print(f"Shannon optimum      = {shannon:.6f} nats")
print(f"excess = inference gap = {net - shannon:.6f} nats   (Problem L0.1 pattern)")
assert abs(net - (-(float((q_bad * np.log(p_x_given_z[:, 0])).sum()) - kl(q_bad, p_z)))) < 1e-12
assert abs(net - shannon - kl(q_bad, posterior)) < 1e-12

naive two-part cost  = 1.530135 nats
bits-back refund     = 0.693147 nats
net code length      = 0.836988 nats
Shannon optimum      = 0.693147 nats
excess = inference gap = 0.143841 nats   (Problem L0.1 pattern)


### Problem L1.6 — The optimal contrastive critic

**Statement.** Derive the minimizer of the InfoNCE loss over all functions $f(x, y)$ and explain
what a trained critic's logits estimate.

**Intuition.** The contrastive task is a $K$-way classification whose Bayes rule is a density
ratio, so the critic that wins is the log of that ratio.

**Solution.**

*Step 1 (the true posterior).* With one positive and $K-1$ negatives,

$$
\Pr\left[j \mid x, y_{1:K}\right] \propto p(y_j \mid x)\prod_{k \neq j} p(y_k) \propto \frac{p(y_j \mid x)}{p(y_j)},
$$

after dividing by the common factor $\prod_k p(y_k)$.

*Step 2 (propriety).* InfoNCE is the cross-entropy between this posterior and the model softmax,
and the logarithmic score is strictly proper, so the loss is minimized exactly when the two agree.

*Step 3 (solve).* Softmax is invariant to additive shifts, so

$$
f^{\star}(x, y) = \log\frac{p(y \mid x)}{p(y)} + c(x) = \operatorname{PMI}(x, y) + c(x).
$$

*Step 4 (reading logits).* Differences of logits for two candidates estimate
$\log\frac{p(y_1 \mid x)p(y_2)}{p(y_2 \mid x)p(y_1)}$, a calibrated relevance ratio; absolute
logits are not comparable across anchors because of the free $c(x)$. This is Proof 5.8.

$$
\boxed{f^{\star}(x, y) = \operatorname{PMI}(x, y) + c(x)}
$$

**Key takeaway.** Contrastive training is density-ratio estimation, which is why its scores
retrieve well and why they need per-anchor normalization before being read as probabilities.

In [12]:
M_sym, flip = 4, 0.05
channel = np.full((M_sym, M_sym), flip)
np.fill_diagonal(channel, 1.0 - (M_sym - 1) * flip)
p_x_sym = np.ones(M_sym) / M_sym
p_y_sym = p_x_sym @ channel
rho = channel / p_y_sym[None, :]
print("density ratio rho(x, y) = p(y|x)/p(y):")
print(rho)
print(f"\noptimal critic values log rho: diagonal {np.log(rho[0, 0]):.6f}, "
      f"off-diagonal {np.log(rho[0, 1]):.6f}")
shifted = np.log(rho) + rng.normal(size=(M_sym, 1))
softmax_ref = np.exp(np.log(rho)) / np.exp(np.log(rho)).sum(axis=1, keepdims=True)
softmax_shift = np.exp(shifted) / np.exp(shifted).sum(axis=1, keepdims=True)
print(f"max |softmax(log rho) - softmax(log rho + c(x))| = "
      f"{np.abs(softmax_ref - softmax_shift).max():.2e}   (the shift is invisible)")
assert np.allclose(softmax_ref, softmax_shift)
assert abs(float((p_y_sym * rho[0]).sum()) - 1.0) < 1e-12

density ratio rho(x, y) = p(y|x)/p(y):
[[3.4 0.2 0.2 0.2]
 [0.2 3.4 0.2 0.2]
 [0.2 0.2 3.4 0.2]
 [0.2 0.2 0.2 3.4]]

optimal critic values log rho: diagonal 1.223775, off-diagonal -1.609438
max |softmax(log rho) - softmax(log rho + c(x))| = 1.11e-16   (the shift is invisible)


### Problem L1.7 — Gaussian rate-distortion and reverse water-filling

**Statement.** Derive $R(D) = \tfrac12\log(\sigma^2/D)$ for a Gaussian source under squared error
by exhibiting the optimal test channel, then apply reverse water-filling to independent
components with variances $(9, 4, 1, 0.25)$ at water level $\theta = 1$.

**Intuition.** Code each component down to the water level and leave the ones already below it
alone.

**Solution.**

*Step 1 (the test channel).* Let $\hat{X} \sim \mathcal{N}(0, \sigma^2 - D)$ and
$V \sim \mathcal{N}(0, D)$ be independent and set $X = \hat{X} + V$. Then $X$ has the right law,
$\mathbb{E}\left[(X - \hat{X})^2\right] = D$, and

$$
I(X; \hat{X}) = h(X) - h(V) = \tfrac12\log\left(2\pi e \sigma^2\right) - \tfrac12\log(2\pi e D) = \tfrac12\log\frac{\sigma^2}{D}.
$$

*Step 2 (nothing does better).* Proof 5.4 Step 2 shows every test channel with distortion at most
$D$ has $I \ge \tfrac12\log(\sigma^2/D)$, so the minimum is attained.

*Step 3 (water-filling).* For independent components the problem separates, and the optimal
allocation is $D_i = \min(\theta, \sigma_i^2)$ with $\theta$ chosen to meet the total budget. At
$\theta = 1$,

$$
D = (1,\ 1,\ 1,\ 0.25), \qquad \sum_i D_i = 3.25 .
$$

*Step 4 (the rate).* Only components above the water level are coded:

$$
R = \tfrac12\log 9 + \tfrac12 \log 4 + 0 + 0 = \tfrac12\log 36 = \log 6 = 1.791759 \text{ nats} = 2.584963 \text{ bits}.
$$

$$
\boxed{R(D) = \tfrac12\log\frac{\sigma^2}{D}; \quad \text{water-filling at } \theta = 1 \text{ gives } D = 3.25, \; R = \log 6 \text{ nats}}
$$

**Key takeaway.** The component with variance below the water level is not coded at all — the
exact ancestor of dropping small principal components.

In [13]:
variances = np.array([9.0, 4.0, 1.0, 0.25])
theta = 1.0
D_alloc = np.minimum(theta, variances)
R_alloc = 0.5 * np.log(np.maximum(variances / D_alloc, 1.0))
print(f"variances       = {variances}")
print(f"distortions D_i = {D_alloc}   total D = {D_alloc.sum():.4f}")
print(f"rates R_i       = {R_alloc}   total R = {R_alloc.sum():.6f} nats"
      f" = {R_alloc.sum() * NAT_TO_BIT:.6f} bits")
assert abs(D_alloc.sum() - 3.25) < 1e-12
assert abs(R_alloc.sum() - np.log(6.0)) < 1e-12

# a direct check of the scalar formula through the test channel
sigma2, D_one = 4.0, 0.75
h = lambda v: 0.5 * np.log(2 * np.pi * np.e * v)
print(f"\nscalar check: h(X) - h(V) = {h(sigma2) - h(D_one):.10f}"
      f"   0.5 log(sigma^2/D) = {0.5 * np.log(sigma2 / D_one):.10f}")
assert abs((h(sigma2) - h(D_one)) - 0.5 * np.log(sigma2 / D_one)) < 1e-12

variances       = [9.   4.   1.   0.25]
distortions D_i = [1.   1.   1.   0.25]   total D = 3.2500
rates R_i       = [1.0986 0.6931 0.     0.    ]   total R = 1.791759 nats = 2.584963 bits

scalar check: h(X) - h(V) = 0.8369882168   0.5 log(sigma^2/D) = 0.8369882168


## L2 — Applications (AI/ML and Physics)

### Problem L2.1 — The RLHF KL budget

**Statement.** An RLHF run uses $\beta = 0.1$ and converges with a measured
$D_{\mathrm{KL}}\left(\pi_\theta \parallel \pi_{\mathrm{ref}}\right) = 8$ nats per response.
(a) Write the optimal policy. (b) Interpret $8$ nats. (c) What happens if $\beta$ is halved?

**Intuition.** Theorem 4.8 says the penalty is not a brake but the definition of the optimum.

**Solution.**

*Step 1 (optimal policy).*

$$
\pi^{\star}(y \mid x) = \frac{\pi_{\mathrm{ref}}(y \mid x)\exp\left(r(x, y)/\beta\right)}{\mathcal{Z}(x)}, \qquad \mathcal{Z}(x) = \mathbb{E}_{\pi_{\mathrm{ref}}}\left[e^{r/\beta}\right],
$$

with optimal value $\beta\log\mathcal{Z}(x)$, a free energy.

*Step 2 (reading 8 nats).* $8$ nats $= 11.5416$ bits: a response from the tuned model needs about
$11.5$ extra bits to encode under the reference than under itself, a typical likelihood ratio of
$e^{8} = 2981$. Pinsker gives only
$\mathrm{TV} \le \sqrt{8/2} = 2$, which is vacuous — at this divergence the two policies are
effectively disjoint on their typical sets.

*Step 3 (halving $\beta$).* The tilt exponent $r/\beta$ doubles, sharpening the optimum towards
high-reward responses, so the realized KL rises and with it the risk of exploiting reward-model
error where $\pi_{\mathrm{ref}}$ is small. On a **finite** action set the growth eventually stops:
the code cell below sweeps $\beta$ on a three-action toy and the KL saturates at
$\log\left(1/\pi_{\mathrm{ref}}(y^{\star})\right) = \log 5 = 1.6094$ nats, the KL of the point
mass on the best action. A language model's action set is effectively unbounded, which is why its
KL keeps climbing and why the standard remedy is an adaptive controller holding a target KL rather
than a fixed $\beta$.

$$
\boxed{\pi^{\star} \propto \pi_{\mathrm{ref}}e^{r/\beta}; \qquad 8 \text{ nats} = 11.5416 \text{ bits}, \; e^{8} = 2981}
$$

**Key takeaway.** Monitoring KL in nats gives a unit-ful, model-independent measure of how far
alignment has moved the policy.

In [14]:
kl_budget = 8.0
print(f"{kl_budget} nats = {kl_budget * NAT_TO_BIT:.4f} bits, likelihood ratio e^KL = {np.exp(kl_budget):.1f}")
print(f"Pinsker bound on total variation: {np.sqrt(kl_budget / 2):.4f}  (vacuous, since TV <= 1)")

pi_ref = np.array([0.5, 0.3, 0.2])
reward = np.array([0.0, 1.0, 3.0])
print("\nrealized KL as beta falls (reference policy and reward fixed):")
for beta in (1.0, 0.5, 0.25, 0.1):
    w = pi_ref * np.exp(reward / beta)
    pi = w / w.sum()
    print(f"  beta = {beta:5.2f}:  pi* = {pi}   KL = {kl(pi, pi_ref):8.5f} nats"
          f"   value = {beta * np.log(w.sum()):.5f}")
kls = []
for beta in (1.0, 0.5, 0.25, 0.1):
    w = pi_ref * np.exp(reward / beta)
    kls.append(kl(w / w.sum(), pi_ref))
print(f"KL grew by factors {np.array(kls[1:]) / np.array(kls[:-1])} as beta halved/quartered")
assert abs(kl_budget * NAT_TO_BIT - 11.541560327111707) < 1e-10
assert kls[1] > kls[0]

8.0 nats = 11.5416 bits, likelihood ratio e^KL = 2981.0
Pinsker bound on total variation: 2.0000  (vacuous, since TV <= 1)

realized KL as beta falls (reference policy and reward fixed):
  beta =  1.00:  pi* = [0.0938 0.1529 0.7533]   KL =  0.73902 nats   value = 1.67384
  beta =  0.50:  pi* = [0.006  0.0266 0.9674]   KL =  1.43404 nats   value = 2.21184
  beta =  0.25:  pi* = [0.     0.0005 0.9995]   KL =  1.60471 nats   value = 2.59777
  beta =  0.10:  pi* = [0. 0. 1.]   KL =  1.60944 nats   value = 2.83906
KL grew by factors [1.9404 1.119  1.0029] as beta halved/quartered


### Problem L2.2 — The Deep Variational Information Bottleneck

**Statement.** Turn the intractable $I(X; Z) - \beta I(Z; Y)$ into a trainable loss, identifying
the bound used on each term and the direction of each bound.

**Intuition.** Bound each mutual information in the direction its own optimization needs, or the
surrogate means nothing.

**Solution.**

*Step 1 (upper-bound the rate).* By Theorem 4.2, for any variational prior $p(z)$,

$$
I(X; Z) \le \mathbb{E}_{p(x)}\left[D_{\mathrm{KL}}\left(q_\phi(z \mid x) \parallel p(z)\right)\right] = R .
$$

An *upper* bound is what is needed, because $I(X; Z)$ is being minimized.

*Step 2 (lower-bound the relevance).* By the Barber-Agakov step of Proof 5.3, for any decoder
$q_\psi(y \mid z)$,

$$
I(Z; Y) \ge H(Y) + \mathbb{E}\left[\log q_\psi(y \mid z)\right],
$$

with slack $\mathbb{E}\left[D_{\mathrm{KL}}\left(p(y \mid z) \parallel q_\psi(y \mid z)\right)\right] \ge 0$.
A *lower* bound is what is needed, because $I(Z; Y)$ is being maximized; $H(Y)$ is a constant of
the dataset and drops out.

*Step 3 (assemble).*

$$
\mathcal{L}_{\mathrm{VIB}} = \mathbb{E}\left[-\log q_\psi(y \mid z)\right] + \frac{1}{\beta}\,\mathbb{E}_{p(x)}\left[D_{\mathrm{KL}}\left(q_\phi(z \mid x) \parallel p(z)\right)\right],
$$

an ordinary classifier with a stochastic bottleneck layer and a Gaussian KL penalty.

*Step 4 (why direction matters).* Both bounds are conservative the right way, so
$\mathcal{L}_{\mathrm{VIB}}$ upper-bounds the true objective up to a constant, and minimizing a
valid upper bound is a sound relaxation. Reversing either bound destroys the guarantee.

$$
\boxed{\mathcal{L}_{\mathrm{VIB}} = \mathbb{E}\left[-\log q_\psi(y \mid z)\right] + \tfrac{1}{\beta} R \;\ge\; \text{IB objective} + \text{const}}
$$

**Key takeaway.** Assembling variational information objectives is a matter of bounding each
mutual information in the direction its optimization requires — the most transferable trick in
this module.

In [15]:
p_x4 = np.ones(4) / 4
p_y1 = np.array([0.9, 0.8, 0.2, 0.1])
p_y_given_x = np.stack([1.0 - p_y1, p_y1], axis=1)
encoder_hard = np.array([[0.0, 1.0], [0.0, 1.0], [1.0, 0.0], [1.0, 0.0]])
p_z = p_x4 @ encoder_hard
p_y_given_z = ((encoder_hard * p_x4[:, None]).T @ p_y_given_x) / p_z[:, None]

I_xz = mutual_information(p_x4[:, None] * encoder_hard)
I_zy = mutual_information(p_z[:, None] * p_y_given_z)
for prior_name, prior in (("uniform (matched)", p_z), ("mismatched", np.array([0.7, 0.3]))):
    R = float(sum(p_x4[i] * kl(encoder_hard[i], prior) for i in range(4)))
    print(f"prior {prior_name:18s}: R = {R:.6f} >= I(X;Z) = {I_xz:.6f}")
    assert R >= I_xz - 8 * EPS

bad_decoder = np.array([[0.5, 0.5], [0.5, 0.5]])
lower = entropy(p_x4 @ p_y_given_x) + float(
    sum(p_x4[i] * encoder_hard[i, k] * float(p_y_given_x[i] @ np.log(bad_decoder[k]))
        for i in range(4) for k in range(2)))
print(f"\nBarber-Agakov lower bound with a uniform decoder: {lower:.6f} <= I(Z;Y) = {I_zy:.6f}")
assert lower <= I_zy + 1e-12

prior uniform (matched) : R = 0.693147 >= I(X;Z) = 0.693147
prior mismatched        : R = 0.780324 >= I(X;Z) = 0.693147

Barber-Agakov lower bound with a uniform decoder: 0.000000 <= I(Z;Y) = 0.270438


### Problem L2.3 — Contrastive batch size, temperature and the ceiling

**Statement.** A SimCLR-style run uses $K = 1024$ in-batch candidates and temperature
$\tau = 0.1$. (a) What is the mutual-information ceiling? (b) How does $\tau$ enter? (c) What does
raising $K$ to $65{,}536$ buy?

**Intuition.** Theorem 4.9 caps the bound at $\log K$; the temperature rescales the critic, not
the cap.

**Solution.**

*Step 1 (ceiling).* $\log 1024 = 6.931472$ nats $= 10$ bits. No encoder can certify more at this
batch size.

*Step 2 (temperature).* The critic is $f(x,y) = \operatorname{sim}(x,y)/\tau$, and by Problem
L1.6 the optimum is $\operatorname{PMI} + c(x)$. With cosine similarities in $[-1, 1]$ the critic
can only represent PMI values in roughly $[-2/\tau, 2/\tau]$, so a small $\tau$ widens that range
and sharpens the softmax onto hard negatives. It does **not** move the $\log K$ ceiling.

*Step 3 (larger $K$).* $\log 65{,}536 = 11.090355$ nats $= 16$ bits. The gain is logarithmic: a
$64$-fold larger pool buys $\log 64 = 4.158883$ nats $= 6$ bits. Memory banks and momentum
encoders obtain that without a $64$-fold larger gradient batch, which is the whole engineering
point.

$$
\boxed{\text{ceiling} = \log K: \; 6.9315 \text{ nats at } K = 1024, \; 11.0904 \text{ nats at } K = 65536}
$$

**Key takeaway.** Batch size sets what the objective *can* certify; temperature sets how the
critic *represents* density ratios. Conflating them means tuning the wrong knob.

In [16]:
for K in (1024, 65536):
    print(f"K = {K:6d}: ceiling = {np.log(K):.6f} nats = {np.log2(K):.4f} bits")
print(f"gain from 1024 -> 65536: {np.log(65536 / 1024):.6f} nats = {np.log2(64):.1f} bits")
assert abs(np.log(1024) - 6.931471805599453) < 1e-12
assert abs(np.log(65536) - 11.090354888959125) < 1e-12
assert abs(np.log(65536) - np.log(1024) - np.log(64)) < 1e-12
for tau in (1.0, 0.1, 0.05):
    print(f"tau = {tau:.2f}: representable PMI range with cosine in [-1, 1] is "
          f"[{-2 / tau:.1f}, {2 / tau:.1f}] nats")

K =   1024: ceiling = 6.931472 nats = 10.0000 bits
K =  65536: ceiling = 11.090355 nats = 16.0000 bits
gain from 1024 -> 65536: 4.158883 nats = 6.0 bits
tau = 1.00: representable PMI range with cosine in [-1, 1] is [-2.0, 2.0] nats
tau = 0.10: representable PMI range with cosine in [-1, 1] is [-20.0, 20.0] nats
tau = 0.05: representable PMI range with cosine in [-1, 1] is [-40.0, 40.0] nats


### Problem L2.4 — Scaling laws read as compression

**Statement.** A model family fits $L(N) = A N^{-\alpha} + L_{\infty}$ in nats per token, with
$L = 2.20$ at $N = 10^{9}$, $L = 1.95$ at $N = 10^{10}$ and a fitted $L_{\infty} = 1.60$. Compute
$\alpha$, the bits-per-token improvement, and the compression consequence on a corpus of
$10^{12}$ tokens.

**Intuition.** Subtract the floor first: only the reducible part obeys the power law.

**Solution.**

*Step 1 (subtract the floor).* The reducible parts are $0.60$ and $0.35$ nats.

*Step 2 (solve for $\alpha$).*
$\frac{0.35}{0.60} = 10^{-\alpha}$, so $\alpha = -\log_{10}(0.583333) = 0.234083$.

*Step 3 (bits per token).* $2.20/\log 2 = 3.173929$ bits and $1.95/\log 2 = 2.813255$ bits, a
saving of $0.360674$ bits per token.

*Step 4 (compression).* On $10^{12}$ tokens that is $3.6067 \times 10^{11}$ bits. Dividing by
$8$ gives $4.508 \times 10^{10}$ bytes, that is **$45.08$ gigabytes** — the corpus shrinking from
$396.7$ GB to $351.7$ GB, a relative saving of $11.36$ percent. (Quoting this as terabytes
overstates it by a factor of a thousand.)

*Step 5 (the floor).* $L_{\infty} = 1.60$ nats $= 2.3083$ bits per token estimates the entropy
rate of the data. By the decomposition
$H_{\times}(p, q) = H(p) + D_{\mathrm{KL}}(p \parallel q)$, scaling shrinks only the KL term; at
$N = 10^{10}$ the model still pays $0.35$ nats of pure model error.

$$
\boxed{\alpha = 0.2341; \quad \text{saving } 0.3607 \text{ bits/token} = 45.08 \text{ GB per } 10^{12} \text{ tokens}; \quad L_{\infty} = 2.3083 \text{ bits/token}}
$$

**Key takeaway.** Scaling laws are compression curves whose asymptote is the data's own entropy;
a loss quoted without its floor hides how much of the error is even addressable.

In [17]:
L1_, N1_ = 2.20, 1e9
L2_, N2_ = 1.95, 1e10
L_inf = 1.60
alpha = -np.log10((L2_ - L_inf) / (L1_ - L_inf)) / np.log10(N2_ / N1_)
bits1, bits2 = L1_ * NAT_TO_BIT, L2_ * NAT_TO_BIT
saving = bits1 - bits2
tokens = 1e12
saved_bytes = saving * tokens / 8
print(f"alpha                    = {alpha:.6f}")
print(f"bits per token           = {bits1:.6f} -> {bits2:.6f}   saving {saving:.6f}")
print(f"bits saved on 1e12 tokens = {saving * tokens:.4e}")
print(f"bytes saved               = {saved_bytes:.4e} = {saved_bytes / 1e9:.2f} GB"
      f" (NOT TB: {saved_bytes / 1e12:.5f} TB)")
print(f"corpus size               = {bits1 * tokens / 8 / 1e9:.1f} GB -> "
      f"{(bits1 - saving) * tokens / 8 / 1e9:.1f} GB   ({saving / bits1:.2%} smaller)")
print(f"irreducible floor         = {L_inf * NAT_TO_BIT:.6f} bits per token")
assert abs(alpha - 0.234083) < 1e-5
assert abs(saved_bytes / 1e9 - 45.084) < 0.01

alpha                    = 0.234083
bits per token           = 3.173929 -> 2.813255   saving 0.360674
bits saved on 1e12 tokens = 3.6067e+11
bytes saved               = 4.5084e+10 = 45.08 GB (NOT TB: 0.04508 TB)
corpus size               = 396.7 GB -> 351.7 GB   (11.36% smaller)
irreducible floor         = 2.308312 bits per token


### Problem L2.5 — InfoGAN's mutual-information regularizer

**Statement.** InfoGAN adds $\lambda\, I\left(c; G(z, c)\right)$ to the GAN objective. Derive the
variational lower bound actually optimized and explain why the auxiliary network is needed.

**Intuition.** "Maximize information with the code" is implemented as "make the code decodable
from the output".

**Solution.**

*Step 1 (the obstacle).* $I(c; X)$ with $X = G(z, c)$ needs the posterior $p(c \mid x)$, which
the generator induces implicitly and which has no closed form.

*Step 2 (Barber-Agakov).* Introduce $Q_\psi(c \mid x)$ and write

$$
I(c; X) = H(c) + \mathbb{E}_{x}\left[D_{\mathrm{KL}}\left(p(c \mid x) \parallel Q_\psi(c \mid x)\right)\right] + \mathbb{E}_{c, x}\left[\log Q_\psi(c \mid x)\right] \ge H(c) + \mathbb{E}_{c, x}\left[\log Q_\psi(c \mid x)\right].
$$

*Step 3 (make it samplable).* The lemma
$\mathbb{E}_{x \sim G}\mathbb{E}_{c' \sim p(c \mid x)}\left[g(c', x)\right] = \mathbb{E}_{c \sim p(c)}\mathbb{E}_{x \sim G(z, c)}\left[g(c, x)\right]$
lets us draw $c$ from the *prior* and $x$ from the generator, giving

$$
L_I(G, Q) = H(c) + \mathbb{E}_{c \sim p(c),\; x \sim G(z, c)}\left[\log Q_\psi(c \mid x)\right] \le I(c; X).
$$

With $p(c)$ fixed, $H(c)$ is constant and the term reduces to a reconstruction loss on the code.

*Step 4 (why $Q$).* $Q_\psi$ *is* the variational posterior: its accuracy sets the bound's
tightness, and its gradient is what forces the generator to make $c$ recoverable from the image.

$$
\boxed{L_I = H(c) + \mathbb{E}\left[\log Q_\psi(c \mid x)\right] \le I(c; X), \quad \text{gap} = \mathbb{E}\left[D_{\mathrm{KL}}\left(p(c \mid x) \parallel Q_\psi\right)\right]}
$$

**Key takeaway.** The Barber-Agakov bound is what licenses translating an information objective
into a decodability objective.

In [18]:
n_codes, n_out = 3, 4
p_c = np.array([0.5, 0.3, 0.2])
gen = rng.dirichlet(np.ones(n_out), size=n_codes)      # p(x | c)
joint_cx = p_c[:, None] * gen
p_out = joint_cx.sum(axis=0)
true_post = joint_cx / p_out[None, :]
I_true = mutual_information(joint_cx)
print(f"true I(c; X)      = {I_true:.6f} nats")
for label, Q in (("exact posterior", true_post),
                 ("uniform Q", np.full((n_codes, n_out), 1 / n_codes)),
                 ("random Q", rng.dirichlet(np.ones(n_codes), size=n_out).T)):
    bound = entropy(p_c) + float((joint_cx * np.log(Q)).sum())
    gap = float(sum(p_out[j] * kl(true_post[:, j], Q[:, j]) for j in range(n_out)))
    print(f"  {label:16s}: L_I = {bound:9.6f}   gap = {gap:.6f}   L_I + gap = {bound + gap:.6f}")
    assert bound <= I_true + 1e-12
    assert abs(bound + gap - I_true) < 1e-10

true I(c; X)      = 0.105153 nats
  exact posterior : L_I =  0.105153   gap = 0.000000   L_I + gap = 0.105153
  uniform Q       : L_I = -0.068959   gap = 0.174113   L_I + gap = 0.105153
  random Q        : L_I = -0.810967   gap = 0.916120   L_I + gap = 0.105153


### Problem L2.6 — Weight noise as description length

**Statement.** Hinton and van Camp minimize
$\mathbb{E}_{q(w)}\left[-\log p(\mathcal{D} \mid w)\right] + D_{\mathrm{KL}}\left(q(w) \parallel p(w)\right)$
over a Gaussian posterior on weights. Show this is a description length, compute the KL for one
weight with $q = \mathcal{N}(\mu, \sigma^2)$ and $p = \mathcal{N}(0, s^2)$, and state the
PAC-Bayes theorem the same KL appears in — with its hypotheses.

**Intuition.** Bits-back applied to weights instead of latents: precision is what you pay for.

**Solution.**

*Step 1 (it is bits-back).* Replace the latent $z$ of Problem L1.5 by the weight vector $w$: the
sender encodes $w$ under the shared prior, encodes the data under $p(\mathcal{D} \mid w)$, and
takes back $-\log q(w)$ nats. The net cost is the variational free energy, so "noisy weights with
a KL penalty" is literally "describe the model, then describe the data given it".

*Step 2 (the per-weight KL).*

$$
D_{\mathrm{KL}}\left(\mathcal{N}(\mu, \sigma^2) \parallel \mathcal{N}(0, s^2)\right) = \ln\frac{s}{\sigma} + \frac{\sigma^2 + \mu^2}{2 s^2} - \frac12 .
$$

A weight left at the prior ($\mu = 0$, $\sigma = s$) costs $0$ nats; a precisely tuned weight
($\sigma \ll s$) costs about $\ln(s/\sigma)$ nats. Precision is paid for in bits.

*Step 3 (consequences).* Pruning, quantization and weight decay all reappear as ways to cut this
cost: a pruned weight costs nothing, a coarsely quantized weight costs few bits, and $L_2$
regularization is the $\mu^2 / 2s^2$ term with $\sigma$ held fixed.

*Step 4 (the theorem, stated properly).* Theorem 4.11: let the loss take values in $[0, 1]$, let
$S$ be $n$ i.i.d. draws, and let the prior $P$ be **fixed before $S$ is seen**. Then with
probability at least $1 - \delta$ over $S$, simultaneously for all posteriors $Q \ll P$,

$$
L_{\mathcal{D}}(Q) \le \widehat{L}_S(Q) + \sqrt{\frac{D_{\mathrm{KL}}(Q \parallel P) + \log\frac{2\sqrt{n}}{\delta}}{2n}} ,
$$

the square-root form obtained from Maurer's kl-form bound by Pinsker. Each hypothesis is
load-bearing: unbounded loss breaks the concentration step, and a prior fitted on $S$ makes the
statement **false**, not merely loose.

$$
\boxed{D_{\mathrm{KL}} = \ln\frac{s}{\sigma} + \frac{\sigma^2 + \mu^2}{2s^2} - \frac12 \text{ nats per weight}}
$$

**Key takeaway.** MDL and PAC-Bayes charge for the *bits of precision* a network needs, not for
its parameter count — which is why heavily overparameterized but compressible networks generalize.

In [19]:
def weight_kl(mu, sigma, s):
    return np.log(s / sigma) + (sigma ** 2 + mu ** 2) / (2 * s ** 2) - 0.5


print("per-weight description cost (nats), prior sigma s = 1")
for mu, sigma in ((0.0, 1.0), (0.3, 0.05), (0.0, 0.01), (2.0, 1.0)):
    print(f"  mu = {mu:4.2f}, sigma = {sigma:5.3f}:  KL = {weight_kl(mu, sigma, 1.0):9.6f} nats"
          f" = {weight_kl(mu, sigma, 1.0) * NAT_TO_BIT:8.4f} bits")
assert abs(weight_kl(0.0, 1.0, 1.0)) < 1e-15
assert abs(weight_kl(0.3, 0.05, 1.0) - 2.541982273553991) < 1e-12

n, delta = 50_000, 0.05
print(f"\nPAC-Bayes square-root term, n = {n}, delta = {delta}")
for d_kl in (10.0, 100.0, 1000.0):
    slack = np.sqrt((d_kl + np.log(2 * np.sqrt(n) / delta)) / (2 * n))
    print(f"  KL(Q || P) = {d_kl:7.1f} nats:  slack = {slack:.5f}")
assert np.sqrt((10.0 + np.log(2 * np.sqrt(n) / delta)) / (2 * n)) < 0.02

per-weight description cost (nats), prior sigma s = 1
  mu = 0.00, sigma = 1.000:  KL =  0.000000 nats =   0.0000 bits
  mu = 0.30, sigma = 0.050:  KL =  2.541982 nats =   3.6673 bits
  mu = 0.00, sigma = 0.010:  KL =  4.105220 nats =   5.9226 bits
  mu = 2.00, sigma = 1.000:  KL =  2.000000 nats =   2.8854 bits

PAC-Bayes square-root term, n = 50000, delta = 0.05
  KL(Q || P) =    10.0 nats:  slack = 0.01382
  KL(Q || P) =   100.0 nats:  slack = 0.03303
  KL(Q || P) =  1000.0 nats:  slack = 0.10045


### Problem L2.7 — Physics: Landauer's principle and the energy floor of erasure

**Statement.** Landauer's principle says that erasing one bit of information in contact with a
heat bath at temperature $T$ dissipates at least $k_{\mathrm{B}} T \ln 2$ joules. (a) Evaluate
that floor at $T = 300$ K. (b) A training run performs $10^{25}$ floating-point operations, each
of which irreversibly discards about $64$ bits; compare the Landauer floor with a realistic
energy budget of $10^{6}$ kWh. (c) What does bits-back coding say about the floor?

**Intuition.** Erasure is the only step of computation thermodynamics charges for, and the price
per bit is tiny — which is exactly why real machines are nowhere near it.

**Solution.**

*Step 1 (the floor per bit).* With $k_{\mathrm{B}} = 1.380649 \times 10^{-23}$ J/K,

$$
E_{\text{bit}} = k_{\mathrm{B}} T \ln 2 = 1.380649 \times 10^{-23} \cdot 300 \cdot 0.693147 = 2.8710 \times 10^{-21} \text{ J}.
$$

*Step 2 (the run).* $10^{25} \times 64 = 6.4 \times 10^{26}$ bits erased, so the floor is
$6.4 \times 10^{26} \cdot 2.8710 \times 10^{-21} = 1.837 \times 10^{6}$ J $= 0.5104$ kWh.

*Step 3 (the comparison).* A budget of $10^{6}$ kWh is $3.6 \times 10^{12}$ J, which is
$1.96 \times 10^{6}$ times the thermodynamic floor. Landauer is not what limits training; it is
six orders of magnitude below the electricity bill.

*Step 4 (bits-back).* Theorem 4.10's refund is not a bookkeeping trick: the auxiliary bits are
*recovered*, not erased, so they never pay $k_{\mathrm{B}}T\ln 2$. A reversible or bits-back
codec converts what would be erasure into transmission, and thermodynamics charges only for the
difference.

$$
\boxed{E_{\text{bit}}(300\,\mathrm{K}) = 2.8710 \times 10^{-21}\ \mathrm{J}; \quad \text{a } 10^{25}\text{-FLOP run has a floor of } 0.51 \text{ kWh, } 1.96 \times 10^{6} \text{ below its budget}}
$$

**Key takeaway.** Information has a thermodynamic price, and it is the *erased* bits that are
billed — which is why "compression is learning" has a physical as well as a statistical reading.

In [20]:
k_B = 1.380649e-23          # J/K, exact by the 2019 SI definition
T_room = 300.0
E_bit = k_B * T_room * np.log(2.0)
print(f"Landauer floor per bit at {T_room:.0f} K = {E_bit:.6e} J = {E_bit / 1.602176634e-19:.6e} eV")

flops, bits_per_flop = 1e25, 64
bits_erased = flops * bits_per_flop
floor_joules = bits_erased * E_bit
budget_kwh = 1e6
budget_joules = budget_kwh * 3.6e6
print(f"bits erased               = {bits_erased:.3e}")
print(f"Landauer floor            = {floor_joules:.6e} J = {floor_joules / 3.6e6:.6f} kWh")
print(f"realistic budget          = {budget_joules:.3e} J = {budget_kwh:.0e} kWh")
print(f"ratio budget / floor      = {budget_joules / floor_joules:.4e}")
print(f"bits that must be erased per second to dissipate 1 W: {1.0 / E_bit:.4e}")
assert abs(E_bit - 2.870978885078724e-21) < 1e-30
assert abs(floor_joules / 3.6e6 - 0.5103962462) < 1e-6
assert 1e6 < budget_joules / floor_joules < 1e7

Landauer floor per bit at 300 K = 2.870979e-21 J = 1.791924e-02 eV
bits erased               = 6.400e+26
Landauer floor            = 1.837426e+06 J = 0.510396 kWh
realistic budget          = 3.600e+12 J = 1e+06 kWh
ratio budget / floor      = 1.9593e+06
bits that must be erased per second to dissipate 1 W: 3.4831e+20


### Problem L2.8 — Physics: the Boltzmann distribution is a KL-regularized optimum

**Statement.** Show that Theorem 4.8 with $r = -E$, $\beta = k_{\mathrm{B}}T$ and a uniform
reference gives the Boltzmann distribution and the Helmholtz free energy. Then evaluate a
two-level system with gap $\Delta = 0.05$ eV at $T = 300$ K: populations, free energy, internal
energy and entropy.

**Intuition.** "Maximize reward, stay near the reference" and "minimize energy, stay disordered"
are the same variational problem with different words.

**Solution.**

*Step 1 (the dictionary).* Put $r(y) = -E(y)$, $\beta = k_{\mathrm{B}}T$ and
$\pi_{\mathrm{ref}}$ uniform on $n$ states. Theorem 4.8 gives

$$
\pi^{\star}(y) \propto e^{-E(y) / k_{\mathrm{B}}T},
$$

the Boltzmann distribution, and the optimal value
$\beta \log \mathbb{E}_{\pi_{\mathrm{ref}}}\left[e^{r/\beta}\right] = k_{\mathrm{B}}T\log\left(\mathcal{Z}/n\right)$
with $\mathcal{Z} = \sum_y e^{-E(y)/k_{\mathrm{B}}T}$.

*Step 2 (identify the free energy).* Since
$D_{\mathrm{KL}}\left(\pi \parallel \text{uniform}\right) = \log n - H(\pi)$, the objective is

$$
-\mathbb{E}_{\pi}[E] + k_{\mathrm{B}}T\, H(\pi) - k_{\mathrm{B}}T\log n = -\left(U - TS\right) - k_{\mathrm{B}}T\log n,
$$

with $S = k_{\mathrm{B}}H(\pi)$. Maximizing it minimizes $F = U - TS = -k_{\mathrm{B}}T\log\mathcal{Z}$.

*Step 3 (the two-level system).* At $T = 300$ K,
$k_{\mathrm{B}}T = 0.0258520$ eV, so $\Delta / k_{\mathrm{B}}T = 1.934086$ and

$$
p_{\text{excited}} = \frac{e^{-1.934086}}{1 + e^{-1.934086}} = 0.126299, \qquad p_{\text{ground}} = 0.873701 .
$$

*Step 4 (thermodynamics).* $\mathcal{Z} = 1.144556$, so
$F = -k_{\mathrm{B}}T\log\mathcal{Z} = -0.003490$ eV, $U = \Delta\, p_{\text{excited}} = 0.006315$
eV, and $S / k_{\mathrm{B}} = H(\pi) = 0.379290$ nats. The identity $F = U - TS$ closes.

$$
\boxed{\pi^{\star} \propto e^{-E/k_{\mathrm{B}}T}, \quad F = -k_{\mathrm{B}}T\log\mathcal{Z} = -0.003490 \text{ eV}, \quad S/k_{\mathrm{B}} = 0.379290 \text{ nats}}
$$

**Key takeaway.** $\beta$ in an RLHF objective is a temperature and $\beta\log\mathcal{Z}$ is a
free energy — not by analogy but by substitution.

In [21]:
eV = 1.602176634e-19
kT_eV = k_B * T_room / eV
gap_eV = 0.05
x = gap_eV / kT_eV
Z_part = 1.0 + np.exp(-x)
p_excited = np.exp(-x) / Z_part
populations = np.array([1.0 - p_excited, p_excited])
F = -kT_eV * np.log(Z_part)
U = gap_eV * p_excited
S_over_kB = entropy(populations)

print(f"k_B T at {T_room:.0f} K      = {kT_eV:.7f} eV")
print(f"Delta / k_B T          = {x:.6f}")
print(f"populations            = {populations}   (ground, excited)")
print(f"partition function Z   = {Z_part:.6f}")
print(f"free energy F          = {F:.6f} eV")
print(f"internal energy U      = {U:.6f} eV")
print(f"entropy S / k_B        = {S_over_kB:.6f} nats")
print(f"U - T S                = {U - kT_eV * S_over_kB:.6f} eV   (must equal F)")
assert abs(U - kT_eV * S_over_kB - F) < 1e-12

energies = np.array([0.0, gap_eV])
uniform = np.array([0.5, 0.5])
objective = lambda pi: float(-(pi @ energies)) - kT_eV * kl(pi, uniform)
print(f"\nTheorem 4.8 value beta log E_ref[exp(r/beta)] = "
      f"{kT_eV * np.log(float(uniform @ np.exp(-energies / kT_eV))):.8f} eV")
print(f"objective at the Boltzmann law               = {objective(populations):.8f} eV")
for other in (uniform, np.array([0.99, 0.01]), np.array([0.6, 0.4])):
    assert objective(other) <= objective(populations) + 1e-15
print("every competing distribution scores lower, as Theorem 4.8 requires")

k_B T at 300 K      = 0.0258520 eV
Delta / k_B T          = 1.934086
populations            = [0.8737 0.1263]   (ground, excited)
partition function Z   = 1.144556
free energy F          = -0.003490 eV
internal energy U      = 0.006315 eV
entropy S / k_B        = 0.379290 nats
U - T S                = -0.003490 eV   (must equal F)

Theorem 4.8 value beta log E_ref[exp(r/beta)] = -0.01442878 eV
objective at the Boltzmann law               = -0.01442878 eV
every competing distribution scores lower, as Theorem 4.8 requires


## L3 — Challenge Proofs

### Problem L3.1 — Deriving the IB self-consistent equations

**Statement.** Derive the stationarity condition of
$\mathcal{L} = I(X; Z) - \beta I(Z; Y)$ with respect to $p(z \mid x)$, subject to normalization,
and interpret the resulting encoder.

**Intuition.** Differentiate an entropy difference and watch the two constants $+1$ annihilate.

**Solution.**

*Step 1 (Lagrangian).* Work in nats and include multipliers for $\sum_z p(z \mid x) = 1$:

$$
\mathcal{F} = I(X; Z) - \beta I(Z; Y) + \sum_x \lambda(x)\sum_z p(z \mid x).
$$

*Step 2 (rate term).* Write
$I(X; Z) = -\sum_z p(z)\ln p(z) + \sum_{x, z} p(x)p(z \mid x)\ln p(z \mid x)$ with
$p(z) = \sum_{x'} p(x')p(z \mid x')$, so $\partial p(z)/\partial p(z \mid x) = p(x)$. The two
pieces give $-p(x)\left[\ln p(z) + 1\right]$ and $p(x)\left[\ln p(z \mid x) + 1\right]$, whose
constants cancel:

$$
\frac{\partial I(X; Z)}{\partial p(z \mid x)} = p(x)\,\ln\frac{p(z \mid x)}{p(z)} .
$$

There is **no** $+1$ in the final expression; a derivation that keeps one has forgotten the
$p(z)$ dependence.

*Step 3 (relevance term).* With $p(z, y) = \sum_x p(x)p(y \mid x)p(z \mid x)$, the implicit
dependence through $p(y \mid z)$ contributes
$p(z)\,\partial_{p(z \mid x)} \sum_y p(y \mid z) = 0$, leaving

$$
\frac{\partial I(Z; Y)}{\partial p(z \mid x)} = p(x)\sum_y p(y \mid x)\,\ln\frac{p(y \mid z)}{p(y)} .
$$

*Step 4 (set to zero and identify the divergence).* Stationarity gives

$$
\ln\frac{p(z \mid x)}{p(z)} = \beta\sum_y p(y \mid x)\ln\frac{p(y \mid z)}{p(y)} - \tilde{\lambda}(x),
$$

and adding and subtracting $\ln p(y \mid x)$ turns the sum into
$-D_{\mathrm{KL}}\left(p(y \mid x) \parallel p(y \mid z)\right)$ plus an $x$-only term absorbed
into $\tilde{\lambda}(x)$.

*Step 5 (exponentiate).* Coupled with $p(z) = \sum_x p(x)p(z \mid x)$ and
$p(y \mid z) = \frac{1}{p(z)}\sum_x p(x)p(y \mid x)p(z \mid x)$, alternating these three updates
is the Blahut-Arimoto form of the IB algorithm. This is Proof 5.5.

$$
\boxed{p(z \mid x) \propto p(z)\exp\left(-\beta\, D_{\mathrm{KL}}\left(p(y \mid x) \parallel p(y \mid z)\right)\right)}
$$

**Key takeaway.** The optimal bottleneck is a soft clustering at inverse temperature $\beta$,
assigning $x$ to the cluster whose predictive law best matches $x$'s own.

In [22]:
p_x4 = np.ones(4) / 4
p_y1 = np.array([0.9, 0.8, 0.2, 0.1])
p_y_given_x = np.stack([1.0 - p_y1, p_y1], axis=1)
TINY = 1e-300


def ib_step(p_z_given_x, beta):
    p_z = p_x4 @ p_z_given_x
    p_y_given_z = ((p_z_given_x * p_x4[:, None]).T @ p_y_given_x) / np.maximum(p_z[:, None], TINY)
    div = (p_y_given_x[:, None, :]
           * (np.log(np.maximum(p_y_given_x, TINY))[:, None, :]
              - np.log(np.maximum(p_y_given_z, TINY))[None, :, :])).sum(axis=-1)
    logits = np.log(np.maximum(p_z, TINY))[None, :] - beta * div
    logits -= logits.max(axis=1, keepdims=True)
    out = np.exp(logits)
    return out / out.sum(axis=1, keepdims=True)


beta = 5.0
p_z_given_x = rng.dirichlet(np.ones(4), size=4)
for _ in range(5000):
    nxt = ib_step(p_z_given_x, beta)
    if np.abs(nxt - p_z_given_x).max() < 1e-14:
        p_z_given_x = nxt
        break
    p_z_given_x = nxt
print(f"fixed point reached; ||T(p) - p||_inf = {np.abs(ib_step(p_z_given_x, beta) - p_z_given_x).max():.2e}")
print("encoder p(z | x) at beta = 5 (rows x, columns z):")
print(p_z_given_x)
print(f"I(X;Z) = {mutual_information(p_x4[:, None] * p_z_given_x):.6f} nats")
assert np.abs(ib_step(p_z_given_x, beta) - p_z_given_x).max() < 1e-12

# the objective decreases along the iteration, from a fresh random start
p_iter = rng.dirichlet(np.ones(4), size=4)
objectives = []
for _ in range(60):
    p_z = p_x4 @ p_iter
    p_y_given_z = ((p_iter * p_x4[:, None]).T @ p_y_given_x) / np.maximum(p_z[:, None], TINY)
    objectives.append(mutual_information(p_x4[:, None] * p_iter)
                      - beta * mutual_information(p_z[:, None] * p_y_given_z))
    p_iter = ib_step(p_iter, beta)
objectives = np.array(objectives)
print(f"\nobjective: first {objectives[0]:.6f} -> last {objectives[-1]:.6f}")
print(f"largest increase along the iteration: {np.diff(objectives).max():.2e} (must be <= 0 up to rounding)")
assert np.diff(objectives).max() < 1e-12

fixed point reached; ||T(p) - p||_inf = 4.44e-16
encoder p(z | x) at beta = 5 (rows x, columns z):
[[0.4999 0.0004 0.4991 0.0006]
 [0.4976 0.0021 0.4967 0.0036]
 [0.0029 0.3712 0.0029 0.623 ]
 [0.0005 0.373  0.0005 0.626 ]]
I(X;Z) = 0.671388 nats

objective: first 0.066575 -> last -0.662353
largest increase along the iteration: 9.99e-16 (must be <= 0 up to rounding)


### Problem L3.2 — The information curve is concave and its slope is $1/\beta$

**Statement.** Let $\mathcal{I}(R) = \max\left\lbrace I(Z; Y) : I(X; Z) \le R \right\rbrace$ under
the Markov chain $Y \to X \to Z$. Prove that $\mathcal{I}$ is non-decreasing and concave, that it
saturates at $I(X; Y)$, that the optimal $\beta$ satisfies $\mathcal{I}'(R) = 1/\beta$, and that
no $\beta \lt 1$ gives a non-trivial optimum.

**Intuition.** Time-sharing between two codebooks makes the achievable region convex, and a
concave frontier has a well-behaved multiplier.

**Solution.**

*Step 1 (non-decreasing).* The feasible set for $R_2 \gt R_1$ contains that for $R_1$, so the
maximum cannot decrease.

*Step 2 (two ceilings).* The data-processing inequality on $Y \to X \to Z$ gives **both**
$I(Z; Y) \le I(X; Y)$ and $I(Z; Y) \le I(X; Z)$, so
$\mathcal{I}(R) \le \min\left(R, I(X; Y)\right)$. Taking $Z = X$ costs rate $H(X)$ and attains
$I(X; Y)$, so the horizontal ceiling is reached.

*Step 3 (concavity by time sharing).* Let $p_1, p_2$ achieve $(R_1, \mathcal{I}(R_1))$ and
$(R_2, \mathcal{I}(R_2))$ on disjoint codebooks, and let $S$ be an independent coin with
$\Pr[S = 1] = \lambda$ selecting which encoder is used. Disjointness makes $S$ a function of $Z$,
so

$$
I(X; Z) = I(X; S) + I(X; Z \mid S) = \lambda R_1 + (1-\lambda)R_2,
$$

$$
I(Z; Y) = I(S; Y) + I(Z; Y \mid S) = \lambda \mathcal{I}(R_1) + (1-\lambda)\mathcal{I}(R_2),
$$

since $I(X; S) = I(S; Y) = 0$. The mixed pair is achievable, which is concavity.

*Step 4 (the slope).* Concavity removes the duality gap, so for each interior $R$ there is a
multiplier $\mu = \mathcal{I}'(R)$ with
$\mathcal{I}(R) - \mu R = \max\left\lbrace I(Z;Y) - \mu I(X;Z)\right\rbrace$. Scaling the bracket
by $1/\mu$ turns it into $-\mathcal{L}_{\mathrm{IB}}/\mu$ with $\beta = 1/\mu$, so the IB
minimizer at parameter $\beta$ sits where $\mathcal{I}'(R) = 1/\beta$.

*Step 5 (why $\beta \lt 1$ is trivial).* Step 2 gives $I(Z; Y) \le I(X; Z)$ for **every**
encoder, so for $\beta \lt 1$

$$
\mathcal{L}_{\mathrm{IB}} = I(X; Z) - \beta I(Z; Y) \ge (1 - \beta) I(X; Z) \ge 0,
$$

with the value $0$ attained by the constant encoder. Equivalently, the curve's slope never
exceeds $1$, so $1/\beta = \mathcal{I}'(R) \le 1$ forces $\beta \ge 1$. The condition is
necessary but not sufficient: for the joint distribution of Example 6.5 the first genuine split
happens only near $\beta = 2$.

$$
\boxed{\mathcal{I} \text{ concave, non-decreasing}, \; \mathcal{I}(R) \le \min(R, I(X;Y)), \; \mathcal{I}'(R) = 1/\beta, \; \beta \ge 1}
$$

**Key takeaway.** The IB curve is a rate-distortion curve in disguise; concavity is what makes
$\beta$ a well-behaved dial and guarantees every frontier point is reachable by some multiplier.

In [23]:
def ib_solve(beta, n_z=4, max_iter=60_000, tol=1e-15):
    p = rng.dirichlet(np.ones(n_z), size=4)
    prev = None
    for _ in range(max_iter):
        p = ib_step(p, beta)
        I_xz = mutual_information(p_x4[:, None] * p)
        if prev is not None and abs(I_xz - prev) < tol:
            break
        prev = I_xz
    p_z = p_x4 @ p
    p_y_given_z = ((p * p_x4[:, None]).T @ p_y_given_x) / np.maximum(p_z[:, None], TINY)
    return I_xz, mutual_information(p_z[:, None] * p_y_given_z)


I_XY = mutual_information(p_x4[:, None] * p_y_given_x)
pts = np.array([ib_solve(b) for b in (2.5, 3.0, 4.0, 5.0, 6.0, 8.0, 12.0, 30.0)])
print(f"I(X;Y) = {I_XY:.6f} nats\n")
print("  I(X;Z)    I(Z;Y)   both ceilings respected?")
for a, c in pts:
    print(f"{a:9.6f} {c:9.6f}   I(Z;Y) <= I(X;Z): {c <= a + 1e-12}   I(Z;Y) <= I(X;Y): {c <= I_XY + 1e-12}")
    assert c <= a + 1e-12 and c <= I_XY + 1e-12

second = np.diff(pts[:, 1]) / np.diff(pts[:, 0])
print(f"\nsecant slopes along the curve: {second}")
print(f"monotonically decreasing (concavity): {bool(np.all(np.diff(second) < 0))}")
assert np.all(np.diff(second) < 0)
for b in (3.0, 5.0, 8.0):
    lo, hi = ib_solve(b - 0.05), ib_solve(b + 0.05)
    slope = (hi[1] - lo[1]) / (hi[0] - lo[0])
    print(f"beta = {b:4.1f}: measured slope {slope:.6f} vs 1/beta = {1 / b:.6f}")
    assert abs(slope - 1 / b) * b < 0.01
print(f"beta = 0.8 gives I(X;Z) = {ib_solve(0.8)[0]:.2e}  (trivial, as Step 5 requires)")
assert ib_solve(0.8)[0] < 1e-6

I(X;Y) = 0.280404 nats

  I(X;Z)    I(Z;Y)   both ceilings respected?
 0.375327  0.170949   I(Z;Y) <= I(X;Z): True   I(Z;Y) <= I(X;Y): True
 0.522088  0.225093   I(Z;Y) <= I(X;Z): True   I(Z;Y) <= I(X;Y): True
 0.634558  0.258373   I(Z;Y) <= I(X;Z): True   I(Z;Y) <= I(X;Y): True
 0.671388  0.266748   I(Z;Y) <= I(X;Z): True   I(Z;Y) <= I(X;Y): True
 0.684855  0.269240   I(Z;Y) <= I(X;Z): True   I(Z;Y) <= I(X;Y): True
 0.691924  0.270301   I(Z;Y) <= I(X;Z): True   I(Z;Y) <= I(X;Y): True
 0.693121  0.270436   I(Z;Y) <= I(X;Z): True   I(Z;Y) <= I(X;Y): True
 0.693147  0.270438   I(Z;Y) <= I(X;Z): True   I(Z;Y) <= I(X;Y): True

secant slopes along the curve: [0.3689 0.2959 0.2274 0.185  0.1502 0.1126 0.0772]
monotonically decreasing (concavity): True
beta =  3.0: measured slope 0.333491 vs 1/beta = 0.333333
beta =  5.0: measured slope 0.200040 vs 1/beta = 0.200000
beta =  8.0: measured slope 0.125014 vs 1/beta = 0.125000
beta = 0.8 gives I(X;Z) = 1.88e-16  (trivial, as Step 5 requires)


### Problem L3.3 — Why every $K$-sample mutual-information lower bound is capped

**Statement.** Prove that $\mathcal{L}_{\mathrm{NCE}} \ge 0$, hence that $\log K$ caps the
InfoNCE bound, and explain why the limitation is a property of the estimation problem rather than
of this particular loss.

**Intuition.** A $K$-way classifier can be right at most all the time, so it can reveal at most
$\log K$ nats about which candidate was the positive.

**Solution.**

*Step 1 (the loss is non-negative).* $\mathcal{L}_{\mathrm{NCE}}$ is an expected negative
log-probability of an event, and probabilities lie in $(0, 1]$, so it is at least $0$. Therefore
$\log K - \mathcal{L}_{\mathrm{NCE}} \le \log K$ for every critic and every dataset.

*Step 2 (the bound stalls, it does not break).* With the optimal critic the bound behaves like
$\min\left(I(X;Y),\ \approx \log K\right)$: informative when the truth is well below the ceiling,
stalled when it is far above.

*Step 3 (why this is fundamental).* Consider two joint laws on $K$-sample batches: $P_{\text{ind}}$
under which $x$ and $y$ are independent, so $I = 0$; and $P_{\text{dep}}$, a hidden-matching
construction whose dependence is carried by an event of probability $e^{-I}$ per sample. The
total-variation distance between the two $K$-sample laws is $O\left(K e^{-I}\right)$. A valid
high-confidence *lower* bound must report near $0$ under $P_{\text{ind}}$; if it reported more
than $\log K + c$ under $P_{\text{dep}}$ it would distinguish two laws at total-variation distance
$o(1)$, which is impossible.

*Step 4 (the asymmetry).* Sample-based *lower* bounds are capped at $O(\log K)$ nats. *Upper*
bounds such as the rate $R$ of Theorem 4.2 have no such limitation, because they use a known
prior rather than samples. That is why practitioners can honestly report a rate but not a mutual
information.

$$
\boxed{\mathcal{L}_{\mathrm{NCE}} \ge 0 \implies \text{bound} \le \log K; \text{ every } K\text{-sample lower bound is } O(\log K)}
$$

**Key takeaway.** The ceiling belongs to the estimation problem, which is why "we maximize mutual
information" is a statement about an objective and never about a measurement.

In [24]:
from math import comb


def infonce_exact(alphabet, flip, n_candidates):
    """Exact InfoNCE loss with the optimal critic on a symmetric channel."""
    p_hit = 1.0 - (alphabet - 1) * flip
    rho_hit, rho_miss = alphabet * p_hit, alphabet * flip
    total = 0.0
    for prob, rho_pos in ((p_hit, rho_hit), ((alphabet - 1) * flip, rho_miss)):
        for m in range(n_candidates):
            weight = (comb(n_candidates - 1, m) * (1.0 / alphabet) ** m
                      * (1.0 - 1.0 / alphabet) ** (n_candidates - 1 - m))
            denom = rho_pos + m * rho_hit + (n_candidates - 1 - m) * rho_miss
            total += prob * weight * (np.log(denom) - np.log(rho_pos))
    return total


for alphabet, flip in ((4, 0.05), (64, 1e-6)):
    channel = np.full((alphabet, alphabet), flip)
    np.fill_diagonal(channel, 1.0 - (alphabet - 1) * flip)
    I_true = mutual_information(np.ones(alphabet)[:, None] / alphabet * channel)
    print(f"alphabet {alphabet:3d}, true I(X;Y) = {I_true:.6f} nats")
    for K in (2, 4, 16, 64):
        loss = infonce_exact(alphabet, flip, K)
        print(f"   K = {K:3d}:  L_NCE = {loss:8.6f} >= 0,  bound = {np.log(K) - loss:8.6f}"
              f",  ceiling = {np.log(K):8.6f},  bound <= I: {np.log(K) - loss <= I_true + 1e-12}")
        assert loss >= 0.0
        assert np.log(K) - loss <= min(I_true, np.log(K)) + 1e-12

alphabet   4, true I(X;Y) = 0.798793 nats
   K =   2:  L_NCE = 0.370100 >= 0,  bound = 0.323047,  ceiling = 0.693147,  bound <= I: True
   K =   4:  L_NCE = 0.841535 >= 0,  bound = 0.544759,  ceiling = 1.386294,  bound <= I: True
   K =  16:  L_NCE = 2.035464 >= 0,  bound = 0.737125,  ceiling = 2.772589,  bound <= I: True
   K =  64:  L_NCE = 3.375191 >= 0,  bound = 0.783692,  ceiling = 4.158883,  bound <= I: True
alphabet  64, true I(X;Y) = 4.157950 nats
   K =   2:  L_NCE = 0.010887 >= 0,  bound = 0.682260,  ceiling = 0.693147,  bound <= I: True
   K =   4:  L_NCE = 0.032406 >= 0,  bound = 1.353889,  ceiling = 1.386294,  bound <= I: True
   K =  16:  L_NCE = 0.155693 >= 0,  bound = 2.616896,  ceiling = 2.772589,  bound <= I: True
   K =  64:  L_NCE = 0.567915 >= 0,  bound = 3.590968,  ceiling = 4.158883,  bound <= I: True


### Problem L3.4 — The ELBO isoline and the anatomy of posterior collapse

**Statement.** Using $D + R \ge H(X)$, explain why a VAE with a powerful autoregressive decoder
can attain the optimal ELBO at $R = 0$, and derive the condition under which increasing decoder
capacity causes collapse.

**Intuition.** The objective's level sets are parallel to the constraint boundary, so the
optimizer is free to slide along it.

**Solution.**

*Step 1 (the feasible region).* By Theorem 4.3,
$H(X) - D \le I(X; Z) \le R$, hence $D + R \ge H(X)$. The average ELBO is $-(D + R)$, so **every**
point of the line $D + R = H(X)$ achieves the same optimal value $-H(X)$.

*Step 2 (two extreme optima).* The autoencoding corner has $R = H(X)$, $D = 0$: the latent carries
the whole message. The collapse corner has $R = 0$, $D = H(X)$: $I(X; Z) \le R = 0$ so the latent
is independent of $x$, and the decoder alone models $p(x)$ perfectly, paying exactly the entropy.
Both are ELBO-optimal; the objective is indifferent between a representation and none at all.

*Step 3 (the collapse condition).* Let $D_{\min}(R)$ be the best distortion the decoder family
can reach at rate $R$. Collapse is globally optimal exactly when $D_{\min}(0) = H(X)$. An
autoregressive decoder $p_\theta(x \mid z) = \prod_t p_\theta(x_t \mid x_{\lt t}, z)$ can model
$p(x)$ arbitrarily well while ignoring $z$, so $D_{\min}(0) \to H(X)$ as its capacity grows. And
by Problem L1.2 the KL gradient vanishes at $(\mu, \sigma) = (0, 1)$, so once the optimizer
arrives there is no restoring force.

*Step 4 (remedies, each breaking one step).* $\beta \lt 1$ or free bits tilt the level sets;
weakening the decoder raises $D_{\min}(0)$ above $H(X)$ and removes the corner from the feasible
set; KL annealing makes the encoder informative before the rate is priced; an explicit rate target
fixes $R = R_0 \gt 0$ and lets the multiplier adapt.

$$
\boxed{D + R \ge H(X), \text{ ELBO constant on } D + R = H(X); \quad \text{collapse} \iff D_{\min}(0) = H(X)}
$$

**Key takeaway.** Posterior collapse is not an optimization bug but a property of the objective's
level sets: the ELBO scores compression of the data and is silent about whether the compression
lives in the code or in the decoder.

In [25]:
H_X = np.log(2.0)
corners = {"collapse (R = 0)": (0.0, H_X),
           "autoencoding (D = 0)": (H_X, 0.0),
           "interior, matched prior": (0.130812035941137, 0.5623351446188083)}
print(f"H(X) = {H_X:.6f} nats\n")
for name, (R, D) in corners.items():
    print(f"{name:26s}: R = {R:.6f}  D = {D:.6f}  D + R = {D + R:.15f}  ELBO = {-(D + R):.6f}")
    assert abs(D + R - H_X) < 8 * EPS
print("\nall three points share the same optimal ELBO, so the objective cannot rank them")
infeasible = (0.20, 0.20)
print(f"an infeasible point R = {infeasible[0]}, D = {infeasible[1]}: D + R = {sum(infeasible):.4f}"
      f" < H(X) = {H_X:.6f}")
assert sum(infeasible) < H_X

H(X) = 0.693147 nats

collapse (R = 0)          : R = 0.000000  D = 0.693147  D + R = 0.693147180559945  ELBO = -0.693147
autoencoding (D = 0)      : R = 0.693147  D = 0.000000  D + R = 0.693147180559945  ELBO = -0.693147
interior, matched prior   : R = 0.130812  D = 0.562335  D + R = 0.693147180559945  ELBO = -0.693147

all three points share the same optimal ELBO, so the objective cannot rank them
an infeasible point R = 0.2, D = 0.2: D + R = 0.4000 < H(X) = 0.693147


### Problem L3.5 — The rate-distortion converse

**Statement.** Let $X^n$ be i.i.d. $p(x)$ and let a code map $X^n$ into one of $2^{nR}$ indices
and back to $\hat{X}^n$ with $\mathbb{E}\left[\frac{1}{n}\sum_i d(X_i, \hat{X}_i)\right] \le D$.
Prove that $R \ge R(D)$, with $R(D)$ the informational rate-distortion function of Definition 3.3.

**Intuition.** The index carries at most $nR$ nats, mutual information cannot exceed it, and
convexity of $R(D)$ turns a per-letter statement into a block statement.

**Solution.**

*Step 1 (the index bounds the information).* Let $W$ be the index. Since $\hat{X}^n$ is a function
of $W$ and $W$ takes at most $2^{nR}$ values (measuring $R$ in the same units as the logarithm),

$$
nR \ge H(W) \ge H\left(\hat{X}^n\right) \ge I\left(X^n; \hat{X}^n\right).
$$

*Step 2 (expand as a difference of entropies).*

$$
I\left(X^n; \hat{X}^n\right) = H(X^n) - H\left(X^n \mid \hat{X}^n\right) = \sum_{i=1}^{n} H(X_i) - \sum_{i=1}^{n} H\left(X_i \mid \hat{X}^n, X^{i-1}\right),
$$

using independence of the $X_i$ for the first sum and the chain rule for the second.

*Step 3 (drop conditioning).* Conditioning cannot increase entropy, so
$H\left(X_i \mid \hat{X}^n, X^{i-1}\right) \le H\left(X_i \mid \hat{X}_i\right)$ and

$$
I\left(X^n; \hat{X}^n\right) \ge \sum_{i=1}^{n}\left[H(X_i) - H\left(X_i \mid \hat{X}_i\right)\right] = \sum_{i=1}^{n} I\left(X_i; \hat{X}_i\right).
$$

*Step 4 (single-letter minimization).* Write $D_i = \mathbb{E}\left[d(X_i, \hat{X}_i)\right]$. The
pair $(X_i, \hat{X}_i)$ is a test channel with distortion $D_i$, so
$I\left(X_i; \hat{X}_i\right) \ge R(D_i)$ by Definition 3.3.

*Step 5 (convexity closes it).* By Theorem 4.4(a) $R$ is convex, so Jensen gives

$$
\frac{1}{n}\sum_{i=1}^{n} R(D_i) \ge R\!\left(\frac{1}{n}\sum_{i=1}^{n} D_i\right) \ge R(D),
$$

the last step because $\frac{1}{n}\sum_i D_i \le D$ and $R$ is non-increasing. Chaining Steps 1
to 5 and dividing by $n$ gives the claim.

$$
\boxed{R \ge R(D) \text{ for every code meeting distortion } D}
$$

**Key takeaway.** The converse needs only three facts — an index carries at most its log-cardinality,
conditioning reduces entropy, and $R(D)$ is convex and non-increasing — which is why Theorem 4.4(a)
had to be proved before the coding theorem could be quoted.

In [26]:
grid = np.linspace(-6.0, 6.0, 161)
w_gauss = np.exp(-grid ** 2 / 2.0)
w_gauss /= w_gauss.sum()
var_g = float((w_gauss * grid ** 2).sum())
R_of = lambda D: 0.5 * np.log(max(var_g / D, 1.0))

print("Step 5 in numbers: Jensen on a convex R(D), Gaussian source")
D_list = np.array([0.05, 0.20, 0.50, 0.80])
mean_D = D_list.mean()
lhs = np.mean([R_of(d) for d in D_list])
rhs = R_of(mean_D)
print(f"  per-letter distortions   = {D_list}")
print(f"  (1/n) sum R(D_i)         = {lhs:.6f} nats")
print(f"  R((1/n) sum D_i)         = {rhs:.6f} nats  at mean D = {mean_D:.4f}")
print(f"  Jensen gap               = {lhs - rhs:.6f} >= 0")
assert lhs >= rhs - 1e-12

print("\nconvexity and monotonicity of R(D) on a grid")
D_axis = np.linspace(0.05, 1.0, 40)
R_axis = np.array([R_of(d) for d in D_axis])
first = np.diff(R_axis) / np.diff(D_axis)
print(f"  R non-increasing : {bool(np.all(np.diff(R_axis) <= 1e-15))}")
print(f"  R convex (slopes increasing) : {bool(np.all(np.diff(first) >= -1e-12))}")
assert np.all(np.diff(R_axis) <= 1e-15)
assert np.all(np.diff(first) >= -1e-12)
print("\nequal allocation is optimal at fixed total distortion, as Step 5 predicts:")
for split in ([0.5, 0.5], [0.3, 0.7], [0.1, 0.9]):
    total = 1.0
    ds = np.array(split) * total
    print(f"  D = {ds}  ->  mean rate = {np.mean([R_of(d) for d in ds]):.6f} nats")
assert np.mean([R_of(d) for d in (0.5, 0.5)]) <= np.mean([R_of(d) for d in (0.3, 0.7)])

Step 5 in numbers: Jensen on a convex R(D), Gaussian source
  per-letter distortions   = [0.05 0.2  0.5  0.8 ]
  (1/n) sum R(D_i)         = 0.690183 nats
  R((1/n) sum D_i)         = 0.474020 nats  at mean D = 0.3875
  Jensen gap               = 0.216163 >= 0

convexity and monotonicity of R(D) on a grid
  R non-increasing : True
  R convex (slopes increasing) : True

equal allocation is optimal at fixed total distortion, as Step 5 predicts:
  D = [0.5 0.5]  ->  mean rate = 0.346574 nats
  D = [0.3 0.7]  ->  mean rate = 0.390162 nats
  D = [0.1 0.9]  ->  mean rate = 0.601986 nats
